<a href="https://colab.research.google.com/github/thedatasense/llm-healthcare/blob/main/models/GPT/gpt-evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q  sqlalchemy pandas psycopg2-binary matplotlib

In [1]:
from ipwhois import IPWhois
from requests import get

ip = get('https://api.ipify.org').text
whois = IPWhois(ip).lookup_rdap(depth=1)
cidr = whois['network']['cidr']
name = whois['network']['name']

print('\n')
print('Provider:  ', name)
print('Public IP: ', ip)
print('CIDRs:     ', cidr)



Provider:   NL-407
Public IP:  216.49.132.215
CIDRs:      216.49.128.0/20


In [13]:
import pandas as pd
from IPython.display import clear_output
from sqlalchemy.engine import create_engine
from openai import OpenAI
import io
import base64
import random
import requests
import torch
from PIL import Image
#from transformers import AutoProcessor,Qwen2_5_VLForConditionalGeneration
#from qwen_vl_utils import process_vision_info
import os
import pandas as pd
from sqlalchemy.engine import create_engine
from transformers import AutoProcessor, BitsAndBytesConfig
import json
import sys,platform
import yaml
from sqlalchemy import text
from IPython.display import clear_output
import time
import json

In [3]:
cnfig_file="/home/bsada1/config.yaml"
def get_from_cnfg(key_path,file_path=cnfig_file):
   try:
       with open(file_path, 'r') as file:
           data = yaml.safe_load(file)

       keys = key_path.split('.')
       value = data
       for key in keys:
           value = value[key]
       return value

   except FileNotFoundError:
       print(f"File {file_path} not found")
   except yaml.YAMLError as e:
       print(f"YAML parsing error: {e}")
   except KeyError:
       print(f"Key path {key_path} not found")
   except Exception as e:
       print(f"Error: {e}")
   return None

In [4]:
os_name=platform.system()
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    from google.colab import userdata
    engine = create_engine(userdata.get('GCP_DB_URL'))
    gem_key=userdata.get('DB_URL')
    oai_key=userdata.get('oai_token')
    b_key_id=userdata.get('BB_KEY_ID')
    b_key=userdata.get('BB_KEY')
    source_folder='/content/drive/MyDrive/Health_Data/MIMIC_JPG/files/'
elif os_name == "Darwin":
    cnfig_file="/Users/bineshkumar/Documents/config.yaml"
    DB_URL = get_from_cnfg("gcp_db_url",cnfig_file)
    engine = create_engine(DB_URL)
    gem_key=get_from_cnfg("gem_token",cnfig_file)
    oai_key=get_from_cnfg("oai_token",cnfig_file)
    b_key_id=get_from_cnfg("bb_token_id",cnfig_file)
    b_key=get_from_cnfg("bb_token",cnfig_file)
    source_folder='/Users/bineshkumar/Documents/mimic-cxr-jpg/2.1.0/files/'
elif os_name == "Linux":
    DB_URL = get_from_cnfg("gcp_db_url",cnfig_file)
    engine = create_engine(DB_URL)
    gem_key=get_from_cnfg("gem_token",cnfig_file)
    oai_key=get_from_cnfg("oai_token",cnfig_file)
    b_key_id=get_from_cnfg("bb_token_id",cnfig_file)
    b_key=get_from_cnfg("bb_token",cnfig_file)
    source_folder=""

In [9]:
def check_duplicate(engine,model_name, question_id):
    query = text("""
        SELECT 1 FROM mimicxp.svu_model_response_evaluation
        WHERE  model_name = :model_name
          AND question_id = :question_id
        LIMIT 1
    """)
    with engine.connect() as conn:
        result = conn.execute(query, {
            "model_name": model_name,
            "question_id": question_id
        }).fetchone()
    return result is not None

In [5]:
def fetch_generation_data(engine):
    import pandas as pd
    import re
    from sqlalchemy import text
    from sqlalchemy.dialects.postgresql.base import PGDialect
    def fake_get_server_version_info(self, connection):
        version_str = connection.execute(text("SELECT version()")).scalar()
        match = re.search(r'v(\d+)\.(\d+)\.(\d+)', version_str)
        if match:
            return tuple(map(int, match.groups()))
        return (13, 0, 0)
    PGDialect._get_server_version_info = fake_get_server_version_info
    query = f"select a.uid, a.question_id, a.question, a.question_category, a.actual_answer, a.model_name, a.model_answer, a.image_link from mimicxp.model_responses_r2 a left join mimicxp.svu_model_response_evaluation b on a.question_id = b.question_id and a.model_name = b.model_name where b.uniqueid is null and a.challenge='surg_vu' ;"
    return pd.read_sql(query, con=engine)



In [6]:
fetch_generation_data(engine)

,uid,question_id,question,question_category,actual_answer,model_name,model_answer,image_link
0,72_730,72_730,Identify the surgical tools in the frame,original,"bipolar forceps or Laparoscopic Vessel Sealer,...",gpt-4o,"{\n ""instruments"": [\n ""Curved dissecting ...",/Users/bineshkumar/Documents/datasets/surgical...
1,72_731,72_731,Identify the surgical tools in the frame,original,tip-up fenestrated grasper or da Vinci Surgica...,gpt-4o,"```json\n{\n ""instruments"": [\n ""Da Vinci ...",/Users/bineshkumar/Documents/datasets/surgical...
2,72_734,72_734,Identify the surgical tools in the frame,original,"needle driver or Laparoscopic trocar, Laparosc...",gpt-4o,"```json\n{\n ""instruments"": [\n ""Laparosco...",/Users/bineshkumar/Documents/datasets/surgical...
3,72_735,72_735,Identify the surgical tools in the frame,original,needle driver or Hemostat Clip,gpt-4o,"```json\n{\n ""instruments"": [\n ""Laparosco...",/Users/bineshkumar/Documents/datasets/surgical...
4,72_738,72_738,Identify the surgical tools in the frame,original,nan(camera in) or Laparoscopic Grasper,gpt-4o,"```json\n{\n ""instruments"": [\n ""Robotic S...",/Users/bineshkumar/Documents/datasets/surgical...
...,...,...,...,...,...,...,...,...
8582,8_482,8_482,Identify the surgical tools in the frame,original,nan(camera in) or Error parsing JSON response,gpt-4o,"```json\n{\n ""instruments"": [\n ""Grasper i...",/Volumes/TVault2/datasets/sugvu24/extracted_fr...
8583,8_483,8_483,Identify the surgical tools in the frame,original,needle driver or Error parsing JSON response,gpt-4o,"```json\n{\n ""instruments"": [""Surgical forcep...",/Volumes/TVault2/datasets/sugvu24/extracted_fr...
8584,8_484,8_484,Identify the surgical tools in the frame,original,nan(camera in) or Error parsing JSON response,gpt-4o,"{\n ""instruments"": []\n}",/Volumes/TVault2/datasets/sugvu24/extracted_fr...
8585,8_485,8_485,Identify the surgical tools in the frame,original,nan(camera in) or Error parsing JSON response,gpt-4o,"```json\n{\n ""instruments"": [\n ""Laparosco...",/Volumes/TVault2/datasets/sugvu24/extracted_fr...


In [8]:
import json

def clean_response(response):
    response = response.strip()
    if response.startswith("```") and response.endswith("```"):
        lines = response.splitlines()
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        response = "\n".join(lines).strip()
    return response

def evaluate_model_answer(original_question, ground_truth, model_answer):
    client = OpenAI(api_key=oai_key)
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a scoring engine for evaluating surgical tool detection in surgical frames. "
                    "For each test case, you are provided with two lists: a ground truth list of surgical instruments present in the frame, "
                    "and a list of instruments predicted by the model. "
                    "Evaluate the model's performance based on the following qualitative scale:\n\n"
                    "1. **Very Poor (1):** No instruments are correctly detected or the predictions are entirely incorrect.\n"
                    "2. **Poor (2):** Only one or a few instruments are correctly detected, with many omissions or false positives.\n"
                    "3. **Moderate (3):** Some instruments are correctly detected, but several important instruments are missed or incorrectly added.\n"
                    "4. **Good (4):** Most instruments are correctly detected with only minor errors or omissions.\n"
                    "5. **Excellent (5):** Nearly perfect detection where all ground truth instruments are correctly identified with no significant errors.\n\n"
                    "Also, identify and list the following:\n"
                    "- **true_positives**: Instruments correctly detected (present in both ground truth and predictions).\n"
                    "- **false_positives**: Instruments incorrectly detected (present in predictions but not in ground truth).\n"
                    "- **false_negatives**: Instruments missed (present in ground truth but not in predictions).\n\n"
                    "When you respond, please provide your answer as valid JSON using the following exact keys:\n"
                    "'qualitative_score', 'true_positives', 'false_positives', 'false_negatives', 'precision', 'recall', and 'remarks'."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Original Question: {original_question}\n\n"
                    f"Ground Truth: {ground_truth}\n\n"
                    f"Model Answer: {model_answer}\n\n"
                    "Please evaluate the surgical tool detection performance and provide the breakdown using the specified JSON keys."
                )
            },
        ],
    )
    content = clean_response(response.choices[0].message.content)
    return json.loads(content)


In [15]:
for index, row in fetch_generation_data(engine).iterrows():
    row_id = row["uid"]
    prompt = row["question"]
    ground_truth = row["actual_answer"]
    model_answer = row["model_answer"]

    clear_output(wait=True)
    print(f"Processing id: {row_id}")

    if check_duplicate(engine, row.get("model_name", ""), row.get("question_id", "")):
        print(f"Duplicate found for combination of uid:{row_id}, model_name, and question_id, skipping.")
        continue

    while True:
        try:
            print(f"Evaluating model answer for id {row_id}...")
            response_json = evaluate_model_answer(prompt, ground_truth, model_answer)
            print(f"Evaluation successful for id {row_id}: {response_json}")

            insert_query = text("""
                INSERT INTO mimicxp.svu_model_response_evaluation (
                     model_name, question_id, ground_truth, model_answer, evaluation_score
                ) VALUES (
                    :model_name, :question_id, :ground_truth, :model_answer, :evaluation_score
                )
            """)
            params = {
                "model_name": row.get("model_name", ""),
                "question_id": row.get("question_id", ""),
                "ground_truth": ground_truth,
                "model_answer": model_answer,
                "evaluation_score": response_json["qualitative_score"]
            }
            with engine.begin() as conn:
                conn.execute(insert_query, params)
            print(f"Record inserted for id {row_id}.")
            break

        except Exception as e:
            print(f"Error for id {row_id}: {e}. Retrying in 10 seconds...")
            time.sleep(10)

Evaluating model answer for id 81_553...
Evaluation successful for id 81_553: {'qualitative_score': 1, 'true_positives': [], 'false_positives': ['Laparoscopic Forceps', 'Laparoscopic Suction Irrigator', 'Laparoscopic Grasper', 'Laparoscopic Scissors'], 'false_negatives': ['needle driver or Large Needle Driver', 'Large SutureCut Needle Driver'], 'precision': 0.0, 'recall': 0.0, 'remarks': 'No instruments from the ground truth were detected, indicating a complete failure in detection.'}
Record inserted for id 81_553.


Add `%load_ext cudf.pandas` before importing pandas to speed up operations using GPU